## OpenRouter Lab 1 - Pydantic v2 response models

Course `1_lab1` flow: completions go through OpenRouter; every assistant message is parsed with **`model_validate_json`** into a **`BaseModel`** (`extra="forbid"`, stripped strings). Set **`OPENROUTER_API_KEY`** in the repo-root `.env`.

In [ ]:
import json
import os
from pathlib import Path
from typing import TypeVar

import httpx
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field

T = TypeVar("T", bound=BaseModel)

In [ ]:
for d in [Path.cwd(), *Path.cwd().parents]:
    env_path = d / ".env"
    if env_path.is_file():
        load_dotenv(env_path, override=True)
        break
else:
    load_dotenv(override=True)

In [ ]:
key = os.getenv("OPENROUTER_API_KEY")
if not key:
    raise RuntimeError("OPENROUTER_API_KEY missing in .env")
base = (os.getenv("OPENROUTER_API_BASE") or os.getenv("OPENROUTER_BASE_URL") or "https://openrouter.ai/api/v1").strip().rstrip("/")
client = OpenAI(
    base_url=base,
    api_key=key,
    http_client=httpx.Client(timeout=httpx.Timeout(120.0, connect=30.0), trust_env=True),
)
MODEL = "openai/gpt-4.1-mini"


def complete(model_cls: type[T], user: str) -> T:
    """One chat completion; body must be JSON matching model_cls."""
    schema = model_cls.model_json_schema()
    system = (
        "Return a single JSON object only (no markdown). "
        f"It must validate against this schema: {json.dumps(schema)}"
    )
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format={"type": "json_object"},
    )
    raw = r.choices[0].message.content
    if not raw:
        raise ValueError("empty completion")
    return model_cls.model_validate_json(raw)

In [ ]:
class StrictJson(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)


class Sum(StrictJson):
    answer: str = Field(min_length=1, max_length=256)


class Puzzle(StrictJson):
    question: str = Field(min_length=8, max_length=2000)


class Solved(StrictJson):
    answer: str = Field(min_length=1, max_length=8000)


class Area(StrictJson):
    area: str = Field(min_length=2, max_length=200)


class Pain(StrictJson):
    pain_point: str = Field(min_length=10, max_length=2000)


class Pitch(StrictJson):
    bullets: list[str] = Field(min_length=1, max_length=6, description="1-6 strings")

In [ ]:
complete(Sum, "2+2=?")

In [ ]:
puzzle = complete(Puzzle, "One hard IQ-style question.")
puzzle

In [ ]:
complete(Solved, puzzle.question)

In [ ]:
biz = complete(Area, "Name one industry where agentic LLMs are a strong fit.")
pain = complete(Pain, f"{biz.area}: one concrete operational pain agents could help with.")
pitch = complete(
    Pitch,
    f"{biz.area} - {pain.pain_point}. Sketch the product in at most six bullets.",
)
biz, pain, pitch